In [47]:
# Install
!pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 117.7 MB/s eta 0:00:00


In [81]:
# Streamlit UI
%%writefile app.py
import streamlit as st
import numpy as np
import joblib
import plotly.graph_objects as go
from tensorflow.keras.models import load_model
from sklearn.cluster import KMeans
st.set_page_config(page_title="Football AI System", layout="wide")
def set_bg():
    bg_url = "https://images.unsplash.com/photo-1656599896998-2fecd1ea0071?q=80&w=1471&auto=format&fit=crop&ixlib=rb-4.1.0"
    st.markdown(f"""
    <style>
    .stApp {{
        background-image: url("{bg_url}");
        background-size: cover;
        background-position: center;
        background-attachment: fixed;
    }}
    .stApp::before {{
        content: "";
        position: fixed;
        width: 100%;
        height: 100%;
        background: rgba(0,0,0,0.75);
        z-index: -1;
    }}
    .block-container {{
        background: transparent !important;
    }}
    label {{
        font-size: 18px !important;
        font-weight: bold !important;
        color: white !important;
    }}
    h1 {{
        color: #00f5ff;
        text-align: center;
        text-shadow: 0 0 20px #00f5ff;
    }}
    </style>
    """, unsafe_allow_html=True)
set_bg()
player_model = load_model("player_model.keras")
team_model = joblib.load("team_model.pkl")
coach_model = joblib.load("coach_model.pkl")
scaler = joblib.load("scaler.pkl")
st.title("⚽ Football Intelligence System")
tab1, tab2, tab3 = st.tabs(["👤 Player", "🏟️ Team", "🧑‍🏫 Coach"])
# HELPERS
def get_archetype(stats):
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    sample = np.array([
        [90,85,70,88,40,75],
        [60,65,85,80,50,70],
        [50,40,60,55,85,80]
    ])
    kmeans.fit(sample)
    return ["🔥 Attacker","🎯 Playmaker","🛡️ Defender"][kmeans.predict([stats])[0]]
def ai_player(stats, rating):
    text = []
    if rating > 85: text.append("Elite player")
    elif rating > 75: text.append("Strong performer")
    else: text.append("Developing player")
    if stats[0] > 80: text.append("Fast")
    if stats[2] > 80: text.append("Great vision")
    if stats[4] > 80: text.append("Defensive strength")
    return " | ".join(text)
def ai_team(inputs, rating):
    attack = (inputs[0] + inputs[1] + inputs[3]) / 3
    defense = (inputs[4] + inputs[5]) / 2
    if attack > defense:
        style = "Attacking Team"
    elif defense > attack:
        style = "Defensive Team"
    else:
        style = "Balanced Team"
    level = "Elite" if rating > 80 else "Competitive" if rating > 70 else "Average"
    return f"{level} team with {style.lower()} tendencies."
def ai_coach(exp, wins, rating):
    if rating > 80:
        return "Elite strategist with strong leadership and consistent results."
    elif rating > 65:
        return "Reliable coach with decent tactical understanding."
    else:
        return "Coach needs improvement in consistency and decision-making."
def radar(players, names):
    labels = ['Pace','Shooting','Passing','Dribbling','Defending','Physical']
    fig = go.Figure()
    for stats, name in zip(players, names):
        fig.add_trace(go.Scatterpolar(
            r=stats + [stats[0]],
            theta=labels + [labels[0]],
            fill='toself',
            name=name
        ))
    fig.update_layout(
        polar=dict(radialaxis=dict(range=[0,100])),
        paper_bgcolor='rgba(0,0,0,0)',
        font=dict(color="white")
    )
    return fig
def fifa_card(rating, label):
    return f"""
    <div style='
        border:2px solid #00f5ff;
        padding:15px;
        border-radius:15px;
        text-align:center;
        background:rgba(0,0,0,0.6);
        box-shadow:0 0 15px #00f5ff;
    '>
    <h3>{label}</h3>
    <h2>⭐ {round(rating,2)}</h2>
    </div>
    """
# PLAYER
with tab1:
    st.header("🎮 Player Intelligence")
    mode = st.radio("Mode", ["Single Player", "⚔️ Compare Players"])
    if mode == "Single Player":
        stats = [st.slider(f,0,100,70) for f in
                 ["Pace","Shooting","Passing","Dribbling","Defending","Physical"]]
        gender = st.selectbox("Gender",["Male","Female"])
        g = 0 if gender=="Male" else 1
        if st.button("Analyze Player"):
            rating = player_model.predict(scaler.transform([stats+[g]]))[0][0]
            st.markdown(fifa_card(rating,"Player"), unsafe_allow_html=True)
            st.info(get_archetype(stats))
            st.write(ai_player(stats, rating))
            st.plotly_chart(radar([stats],["Player"]), use_container_width=True)
    else:
        col1, col2 = st.columns(2)
        players = []
        for i, col in enumerate([col1,col2]):
            with col:
                st.markdown(f"### Player {i+1}")
                stats = [st.slider(f"{f} {i}",0,100,70) for f in
                         ["Pace","Shooting","Passing","Dribbling","Defending","Physical"]]
                players.append(stats)
        if st.button("Compare Players"):
            ratings = [player_model.predict(scaler.transform([p+[0]]))[0][0] for p in players]
            c1, c2 = st.columns(2)
            with c1:
                st.markdown(fifa_card(ratings[0],"Player 1"), unsafe_allow_html=True)
                st.write(ai_player(players[0], ratings[0]))
            with c2:
                st.markdown(fifa_card(ratings[1],"Player 2"), unsafe_allow_html=True)
                st.write(ai_player(players[1], ratings[1]))
            st.success(f"🏆 {'Player 1' if ratings[0]>ratings[1] else 'Player 2'} Wins")
            st.plotly_chart(radar(players,["P1","P2"]), use_container_width=True)
# TEAM
with tab2:
    st.header("🏟️ Team Intelligence")
    inputs = [st.slider(f,0,100,70,key=f) for f in
              ["Pace","Shooting","Passing","Dribbling","Defending","Physical"]]
    if st.button("Analyze Team"):
        rating = team_model.predict([inputs])[0]
        st.success(f"🏆 {round(rating,2)}")
        st.markdown("### 🤖 AI Team Analysis")
        st.info(ai_team(inputs, rating))
# COACH
with tab3:
    st.header("🧠 Coach Intelligence")
    exp = st.slider("Experience",1,20,10)
    wins = st.slider("Wins",10,100,50)
    team = st.slider("Team Rating",50,90,70)
    if st.button("Evaluate Coach"):
        rating = coach_model.predict([[exp,wins,team]])[0]
        st.success(f"🧠 {round(rating,2)}")
        st.markdown("### 🤖 AI Coach Analysis")
        st.info(ai_coach(exp, wins, rating))

Writing app.py


In [82]:
# Run Streamlit
from pyngrok import ngrok
ngrok.set_auth_token("2xga54IR4fN2xmrLn0i10bi0VjL_484W4RVEg9L6N5cd4aMwt")
import subprocess
import time
import os
ngrok.kill()
PORT = 8503
os.system(f"streamlit run app.py --server.port {PORT} &")
time.sleep(5)
public_url = ngrok.connect(PORT)
print("🚀 Streamlit Public URL:", public_url)

🚀 Streamlit Public URL: NgrokTunnel: "https://5af0-34-186-1-146.ngrok-free.app" -> "http://localhost:8503"
